# Generate a list of new strings to match

Now that I have a list of strings from receipts from the previous year, I can improve my matching. 

In [1]:
%pip install -q pandas psycopg2

  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [34 lines of output]
      /tmp/pip-build-env-hyi9indi/overlay/lib/python3.13/site-packages/setuptools/dist.py:759: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
      
              License :: OSI Approved :: GNU Library or Lesser General Public License (LGPL)
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
              ********************************************************************************
      
      !!
        self._finalize_license_expression()
      running egg_info
      writing psycopg2.egg-info/PKG-INFO
      writing dependency_links to psyc

In [10]:

import psycopg2
import pandas as pd

# Connect to PostgreSQL
conn = psycopg2.connect(
    host="localhost",
    database="receipts_app",
    user="postgres",
    password="postgres",
    options="-c search_path=receipts_app"
)
cur = conn.cursor()

In [11]:
receipt_texts_sql = """
SELECT 
    e.id,
    e.product_id,
    rt.text
FROM 
    receipts_app.expense e
JOIN 
    receipts_app.receipt_text rt ON e.receipt_text_id = rt.id
JOIN 
    receipts_app.product p ON e.product_id = p.id;
"""

cur.execute(receipt_texts_sql)

# Fetch results
results = cur.fetchall()

In [ ]:
df = pd.DataFrame(results, columns=[
    'expense_id'
	,'product_id'
	,'receipt_text'
])

In [16]:
df_clean = df.copy()
df_clean['receipt_text'] = df_clean['receipt_text'].str.replace(r'\s+', ' ', regex=True).str.strip()
df_clean = df_clean.drop(columns=['expense_id'])
df_clean = df_clean.drop_duplicates(subset='receipt_rext').reset_index(drop=True)
df_clean

,product_id,receipt_rext
0,98,DEPTOOO1
1,99,DEPTOOO 1
2,97,DEPTOOO5
3,101,PERE TRAPPISTES CHOCOLAT
4,102,TARTINADE DE BLEUETS
...,...,...
395,190,Watkins Organic Paprika
396,44,Rizopia 100% Brown Rice Pasta Spaghetti
397,186,Rizopia 100% Brown Rice Pasta Fettuccine
398,186,Rizopia Organic Brown Rice Pasta Fusilli


In [17]:
df_clean.to_csv('receipt_text_strings.csv', index=False)


In [18]:
# Close connection
cur.close()
conn.close()

In [22]:
import pandas as pd

# Load the cleaned receipt texts from the CSV file
df_loaded = pd.read_csv('data/receipt_texts_clean.csv')

# Display the first few rows of the loaded DataFrame
print(df_loaded.head())


   product_id                receipt_text
0         101    PERE TRAPPISTES CHOCOLAT
1         102        TARTINADE DE BLEUETS
2         102        TARTINADE FRAISES ET
3         103  FAVUZZI CHOCOLAT NOISETTES
4         104       noix DE LUXE AU SIROP


In [24]:
df_loaded = df_loaded.drop_duplicates(subset='receipt_text').reset_index(drop=True)


In [25]:
df_loaded.to_csv('data/receipt_texts_clean.csv', index=False)


In [26]:
# --ALTER TABLE receipts_app.receipt ADD COLUMN receipt_date TIMESTAMP;
# --UPDATE receipts_app.receipt SET receipt_date = '2023-01-01' WHERE receipt_date IS NULL;